# Research Companion v0 — Qwen 4B + Unsloth LoRA Fine-Tuning

This Colab notebook fine-tunes a small Qwen model to behave like a **citation-first research chatbot**.

Goal:

```text
User asks factual/current/research question
→ model decides search is needed
→ model emits a structured tool call
→ backend/search tool returns sources
→ model reads sources
→ model answers with citations and uncertainty
```

This notebook is for **v0 training**, not final production.  
Start small, verify it trains, then scale dataset size and steps.

## 0. Colab runtime setup

In Colab:

```text
Runtime → Change runtime type → Hardware accelerator → T4 GPU
```

Then run the cells top to bottom.

Notes:

- Start with `MAX_SEQ_LENGTH = 2048`.
- Do not attempt 128k/1M context during fine-tuning.
- Qwen3.5 can compile custom kernels slowly on T4. The first training run may feel stuck while compiling.
- For Qwen3.5, Unsloth docs do **not** recommend 4-bit QLoRA because of larger quantization differences, so this notebook starts with 16-bit LoRA.

In [ ]:
# Check GPU
!nvidia-smi

## 1. Install dependencies

This may take a few minutes.  
If Colab asks you to restart runtime after install, restart and continue from the next cell.

In [ ]:
%%capture
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install -U datasets trl accelerate bitsandbytes sentencepiece protobuf huggingface_hub pandas

In [ ]:
import os, json, random, textwrap, torch
from datasets import Dataset, load_dataset
from huggingface_hub import notebook_login

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Optional Hugging Face login

Run this if you want to push the LoRA adapter or GGUF later.

In [ ]:
# Optional. Run only if you want to push results to Hugging Face.
# notebook_login()

## 3. Training config

Main model target: `unsloth/Qwen3.5-4B`.

Fallback: `unsloth/Qwen3-4B-Instruct-2507`.

Use the fallback if Qwen3.5 has install/kernel issues in Colab.

In [ ]:
BASE_MODEL = "unsloth/Qwen3.5-4B"

# Fallback if Qwen3.5 fails on your Colab runtime:
# BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"

OUTPUT_DIR = "research-companion-qwen4b-lora"

# Replace before pushing:
HF_ADAPTER_REPO = "YOUR_HF_USERNAME/research-companion-qwen4b-lora"
HF_GGUF_REPO = "YOUR_HF_USERNAME/research-companion-qwen4b-gguf"

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 80                 # v0 smoke test. Later try 300, 1000, etc.
TRAIN_HOTPOT_SAMPLES = 500     # v0. Later try 5k-50k if runtime allows.
EVAL_SIZE = 0.05
SEED = 3407

# Qwen3.5 docs recommend 16-bit LoRA instead of 4-bit QLoRA.
LOAD_IN_4BIT = False
LOAD_IN_16BIT = True

print("Base model:", BASE_MODEL)

## 4. Load model + LoRA adapter

This uses Unsloth's `FastLanguageModel`.

For T4, keep sequence length small first. If you hit OOM, reduce:

```python
MAX_SEQ_LENGTH = 1024
MAX_STEPS = 30
TRAIN_HOTPOT_SAMPLES = 100
```

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    load_in_16bit = LOAD_IN_16BIT,
    fast_inference = False,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = MAX_SEQ_LENGTH,
)

print("Model and LoRA adapter loaded.")

## 5. Research-chatbot behavior spec

We train the model to follow this contract:

```text
- If the question is factual/current/research-heavy, emit a tool call.
- Search tool returns source blocks.
- Final answer must cite source IDs like [S1], [S2].
- If evidence is weak, say so.
- Do not invent citations.
```

The companion app backend can later parse `<tool_call>{...}</tool_call>`.

In [ ]:
SYSTEM_PROMPT = '''You are Research Companion, a citation-first research chatbot.

Rules:
1. For factual, current, benchmark, product, legal, medical, financial, or research questions, request search/tool use before answering.
2. Use structured tool calls in this exact XML-like wrapper:
   <tool_call>{"name":"web_search","arguments":{"query":"..."}}</tool_call>
3. After sources are provided, answer only from the sources.
4. Cite source IDs inline as [S1], [S2].
5. If sources disagree, explain the disagreement.
6. If the evidence is insufficient, say "I do not have enough evidence" and explain what is missing.
7. Be concise, accurate, and avoid unsupported claims.
'''

def chat_text(messages):
    '''Apply the model chat template when available.'''
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        # fallback plain format
        out = []
        for m in messages:
            out.append(f"{m['role'].upper()}: {m['content']}")
        return "\n\n".join(out)

def make_source_block(sources):
    lines = []
    for s in sources:
        lines.append(f"[{s['id']}] {s['title']}\nURL: {s.get('url','local-dataset')}\n{s['text']}")
    return "\n\n".join(lines)

def make_tool_call(query):
    return '<tool_call>' + json.dumps({
        "name": "web_search",
        "arguments": {"query": query}
    }, ensure_ascii=False) + '</tool_call>'

## 6. Seed tool-use examples

These are small handcrafted examples that teach the exact app protocol.

The premade dataset comes next.

In [ ]:
seed_examples = [
    {
        "user": "Is a local 4B model enough to beat GPT-class models at research?",
        "query": "small language models search tools research assistant accuracy",
        "sources": [
            {
                "id": "S1",
                "title": "Research-agent design note",
                "url": "local-seed://research-agent",
                "text": "Small models can perform well in narrow workflows when paired with retrieval, tools, verification, and citations. They are weaker than frontier models as raw general reasoning systems."
            },
            {
                "id": "S2",
                "title": "RAG reliability note",
                "url": "local-seed://rag",
                "text": "Retrieval-augmented systems reduce hallucination risk when final answers are constrained to retrieved evidence and uncertainty is stated when evidence is insufficient."
            }
        ],
        "answer": "A 4B model is unlikely to beat frontier models as a raw general model. It can still be useful as a research system if it searches, verifies, and answers only from retrieved evidence [S1][S2]."
    },
    {
        "user": "Should I answer current benchmark questions from memory?",
        "query": "latest AI benchmark leaderboard current model scores",
        "sources": [
            {
                "id": "S1",
                "title": "Benchmark freshness note",
                "url": "local-seed://benchmarks",
                "text": "Current model benchmark results change frequently. Answers about latest scores should be verified against current leaderboards or official model pages."
            }
        ],
        "answer": "No. Current benchmark questions should trigger search first because leaderboard positions and model scores can change quickly [S1]."
    },
    {
        "user": "What should the assistant do if sources conflict?",
        "query": "source conflict research answer handling",
        "sources": [
            {
                "id": "S1",
                "title": "Source conflict policy",
                "url": "local-seed://conflict-policy",
                "text": "When sources conflict, a research assistant should state the disagreement, identify which sources support each claim, and avoid presenting either side as settled."
            }
        ],
        "answer": "It should state that the sources conflict, cite which source supports each claim, and avoid treating the issue as settled unless stronger evidence is available [S1]."
    },
]

def seed_to_rows(examples):
    rows = []
    for ex in examples:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": make_tool_call(ex["query"])},
            {"role": "tool", "content": make_source_block(ex["sources"])},
            {"role": "assistant", "content": ex["answer"]},
        ]
        rows.append({"text": chat_text(messages), "source": "seed_tool_use"})
    return rows

rows = seed_to_rows(seed_examples)
print(rows[0]["text"][:1000])

## 7. Load premade research QA data: HotpotQA

HotpotQA is useful because it contains multi-hop QA with supporting Wikipedia context.

We convert examples into:

```text
question → tool call → source blocks → cited final answer
```

This is not perfect yet, but good enough for v0 behavior tuning.

In [ ]:
def build_hotpot_rows(n=500):
    hotpot = load_dataset("hotpot_qa", "distractor", split=f"train[:{n}]")
    out = []

    for ex in hotpot:
        question = ex["question"]
        answer = ex["answer"]

        titles = ex["context"]["title"]
        sentences_list = ex["context"]["sentences"]

        sources = []
        for i, (title, sentences) in enumerate(zip(titles[:6], sentences_list[:6])):
            text = " ".join(sentences[:4])
            text = text.replace("\n", " ").strip()
            if not text:
                continue
            sources.append({
                "id": f"S{i+1}",
                "title": title,
                "url": "hotpotqa://wikipedia-context",
                "text": text[:1200],
            })

        if not sources:
            continue

        # Use supporting fact titles for citation IDs when possible.
        citation_ids = []
        try:
            supporting_titles = set(ex["supporting_facts"]["title"])
            for s in sources:
                if s["title"] in supporting_titles:
                    citation_ids.append(f"[{s['id']}]")
        except Exception:
            pass

        if not citation_ids:
            citation_ids = ["[S1]"]

        cited_answer = (
            f"Based on the provided sources, the answer is: {answer}. "
            f"Evidence: {''.join(citation_ids[:3])}"
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": make_tool_call(question)},
            {"role": "tool", "content": make_source_block(sources)},
            {"role": "assistant", "content": cited_answer},
        ]

        out.append({"text": chat_text(messages), "source": "hotpot_qa"})

    return out

try:
    hotpot_rows = build_hotpot_rows(TRAIN_HOTPOT_SAMPLES)
    rows.extend(hotpot_rows)
    print("Loaded HotpotQA rows:", len(hotpot_rows))
except Exception as e:
    print("HotpotQA load failed. Continuing with seed examples only.")
    print(type(e).__name__, str(e)[:500])

print("Total rows:", len(rows))

## 8. Optional: add FEVER-style verification rows

This is optional because Hugging Face dataset configs can change.

If it loads, it adds examples where the model learns:

```text
supported / refuted / not enough evidence
```

In [ ]:
ADD_FEVER = False  # Set True later after the v0 notebook trains once.

def build_fever_rows(n=300):
    # Dataset names/configs can vary. This block is intentionally wrapped.
    fever = load_dataset("fever", "v1.0", split=f"train[:{n}]")
    out = []
    for ex in fever:
        claim = str(ex.get("claim", "")).strip()
        label = str(ex.get("label", "")).strip()
        if not claim or not label:
            continue

        sources = [{
            "id": "S1",
            "title": "Claim verification record",
            "url": "fever://claim",
            "text": f"Claim: {claim}\nGold label: {label}"
        }]

        answer = (
            f"The claim verification label is {label}. "
            f"This should be treated as a verification task, not an unsupported factual answer [S1]."
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Verify this claim: {claim}"},
            {"role": "assistant", "content": make_tool_call(claim)},
            {"role": "tool", "content": make_source_block(sources)},
            {"role": "assistant", "content": answer},
        ]
        out.append({"text": chat_text(messages), "source": "fever"})
    return out

if ADD_FEVER:
    try:
        fever_rows = build_fever_rows(300)
        rows.extend(fever_rows)
        print("Loaded FEVER rows:", len(fever_rows))
    except Exception as e:
        print("FEVER load failed. Skipping.")
        print(type(e).__name__, str(e)[:500])

print("Total rows:", len(rows))

## 9. Create train/eval dataset

We use a simple `text` column because TRL SFT supports standard text datasets.

In [ ]:
random.seed(SEED)
random.shuffle(rows)

dataset = Dataset.from_list(rows)
dataset = dataset.shuffle(seed=SEED)

if len(dataset) > 20:
    split = dataset.train_test_split(test_size=EVAL_SIZE, seed=SEED)
    train_dataset = split["train"]
    eval_dataset = split["test"]
else:
    train_dataset = dataset
    eval_dataset = None

print(train_dataset)
print("Eval:", eval_dataset)
print("\nPreview:\n")
print(train_dataset[0]["text"][:2000])

## 10. Train

This is a smoke-test run.  
For a real v1, increase:

```python
MAX_STEPS = 300 to 1000+
TRAIN_HOTPOT_SAMPLES = 5000+
```

Do not scale until this small run finishes successfully.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        logging_steps = 1,
        output_dir = "outputs_research_companion",
        optim = "adamw_8bit",
        seed = SEED,
        dataset_num_proc = 1,
        fp16 = True,
        bf16 = False,
        report_to = "none",
    ),
)

trainer.train()

## 11. Save LoRA adapter locally

This saves only the LoRA adapter, not a full merged model.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved adapter to:", OUTPUT_DIR)
!ls -lh {OUTPUT_DIR}

## 12. Quick inference test

This tests whether the model learned the **tool-call-first** behavior.

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def generate(messages, max_new_tokens=512):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Compare Qwythos 9B with Gemini Flash for research tasks. Be accurate."},
]

print(generate(test_messages, max_new_tokens=350))

## 13. Test with provided sources

This simulates what the companion app will do after executing search.

In [ ]:
source_block = '''[S1] Qwythos model card
URL: https://huggingface.co/example/qwythos
Qwythos is a 9B model based on Qwen and advertises long-context capability. Its benchmark claims are self-reported on the model card.

[S2] Public benchmark leaderboard
URL: https://example.com/leaderboard
The public leaderboard does not list Qwythos. Frontier cloud models have independent agent benchmark scores that are much higher than small local models.
'''

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Is Qwythos proven to beat Gemini Flash?"},
    {"role": "tool", "content": source_block},
]

print(generate(test_messages, max_new_tokens=400))

## 14. Push LoRA adapter to Hugging Face

Before running:

```python
HF_ADAPTER_REPO = "your-username/research-companion-qwen4b-lora"
```

Then uncomment and run.

In [ ]:
# Push adapter to Hugging Face.
# Make sure you ran notebook_login() earlier or set HF_TOKEN.

# model.push_to_hub(HF_ADAPTER_REPO)
# tokenizer.push_to_hub(HF_ADAPTER_REPO)

## 15. Optional GGUF export for Ollama / llama.cpp

This can take a long time and may OOM on Colab.

Unsloth supports:

```python
model.save_pretrained_gguf("directory", tokenizer, quantization_method="q4_k_m")
model.push_to_hub_gguf("hf_username/directory", tokenizer, quantization_method="q4_k_m")
```

Run only after the LoRA adapter looks useful.

In [ ]:
DO_GGUF_EXPORT = False

if DO_GGUF_EXPORT:
    GGUF_DIR = "research-companion-qwen4b-Q4_K_M"
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("Saved GGUF to:", GGUF_DIR)

    # Optional push:
    # model.push_to_hub_gguf(
    #     HF_GGUF_REPO,
    #     tokenizer,
    #     quantization_method = "q4_k_m",
    # )

## 16. Suggested next training upgrades

After this notebook works:

```text
1. Increase HotpotQA samples to 5k+
2. Add ASQA / Natural Questions for long-form answers
3. Add RAGTruth / FEVER for hallucination and verification
4. Add synthetic tool-use examples for:
   - search query generation
   - source ranking
   - citation formatting
   - contradiction handling
5. Build a 100-question eval set before training more
```

Expected v0 behavior:

```text
Good:
- emits search tool calls
- cites source IDs
- admits uncertainty more often
- follows research-answer structure

Not yet good:
- true deep research
- source quality ranking
- multi-round tool loops
- fully reliable factuality
```